In [2]:
# Устанавливаем две библиотеки:
# openai   — чтобы обращаться к моделям OpenAI (эмбеддинги + генерация ответа)
# chromadb — векторная база данных, где будут храниться "смысловые" векторы текста
# флаг -q  — тихая установка, без длинного лога на экран
!pip install openai chromadb -q


In [3]:
# Достаём наш секретный API-ключ из хранилища Colab (Secrets),
# не храним его прямо в коде — это небезопасно
from google.colab import userdata
import openai

# получаем сохранённый ключ по его имени "OPENAI_API_KEY"
api_key = userdata.get('OPENAI_API_KEY')

# создаём клиента OpenAI — через него будем делать все запросы к моделям
client = openai.OpenAI(api_key=api_key)

print("Ключ подключен успешно!")

Ключ подключен успешно!


In [4]:
# Открываем окно загрузки файла с компьютера пользователя
from google.colab import files

uploaded = files.upload()

# показываем, какие файлы были загружены
print("Загружены файлы:", list(uploaded.keys()))

Saving 84-0.txt to 84-0.txt
Загружены файлы: ['84-0.txt']


In [5]:
# Функция режет длинный текст на маленькие фрагменты (чанки) фиксированного размера.
# Соседние фрагменты немного "перехлёстываются" (overlap),
# чтобы не терять смысл, если важная мысль оказалась ровно на границе разреза
def split_into_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # сдвигаемся с учётом перекрытия
    return chunks

# читаем содержимое загруженного файла
filename = list(uploaded.keys())[0]  # берём первый загруженный файл
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# режем весь текст книги на фрагменты
chunks = split_into_chunks(text)

# смотрим, сколько получилось фрагментов, и как выглядит первый из них
print(f"Документ разбит на {len(chunks)} фрагментов")
print("Пример первого фрагмента:")
print(chunks[0])


Документ разбит на 1049 фрагментов
Пример первого фрагмента:
*** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chapter 11
 Chapter 12
 Chapter 13
 Chapter 14
 Chapter 15
 Chapter 16
 Chapter 17
 Chapter 18
 Chapter 19
 Chapter 20
 Chapter 21
 Chapter 22
 Chapter 23
 Chapter 24




Letter 1

_To Mrs. Saville, Engla


In [6]:
import chromadb

# создаём (или подключаемся к) векторной базе данных прямо в памяти Colab
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="frankenstein")

# функция превращает любой текст в вектор чисел (эмбеддинг) через модель OpenAI —
# именно по этим векторам потом будет вестись поиск "по смыслу"
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# для теста возьмём не все 1049 фрагментов сразу, а первые 200 —
# чтобы процесс прошёл быстро и не тратить лишние деньги на API
test_chunks = chunks[:200]

# проходим по каждому фрагменту: превращаем в вектор и сохраняем в базу
# вместе с самим текстом фрагмента и названием файла-источника
for i, chunk in enumerate(test_chunks):
    embedding = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"source": filename, "chunk_index": i}]
    )
    # печатаем прогресс каждые 50 фрагментов, чтобы видеть, что процесс не завис
    if (i + 1) % 50 == 0:
        print(f"Обработано {i + 1} из {len(test_chunks)} фрагментов")

print("Готово! Все фрагменты добавлены в векторную базу.")


Обработано 50 из 200 фрагментов
Обработано 100 из 200 фрагментов
Обработано 150 из 200 фрагментов
Обработано 200 из 200 фрагментов
Готово! Все фрагменты добавлены в векторную базу.


In [7]:
# Главная функция всей RAG-системы: получает вопрос и возвращает ответ,
# основанный СТРОГО на содержимом загруженного документа
def ask_question(question, n_results=4):
    # 1. превращаем вопрос в вектор — той же моделью, что и при индексации
    question_embedding = get_embedding(question)

    # 2. ищем в базе самые похожие по смыслу фрагменты
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )
    retrieved_chunks = results['documents'][0]

    # 3. собираем промпт: контекст + вопрос + инструкция отвечать только по контексту
    context = "\n\n---\n\n".join(retrieved_chunks)
    prompt = f"""Ответь на вопрос, используя ТОЛЬКО информацию из приведённого ниже контекста.
Если в контексте нет ответа — честно скажи, что информация недоступна, не выдумывай.

Контекст:
{context}

Вопрос: {question}

Ответ:"""

    # 4. отправляем запрос модели (temperature=0 — чтобы ответы были стабильными,
    #    без "творческой" случайности)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content

    # печатаем сам ответ и список фрагментов-источников, на которые он опирается
    print("ОТВЕТ:")
    print(answer)
    print("\nИСТОЧНИКИ (найденные фрагменты):")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"\n[{i+1}] {chunk[:150]}...")

# тестовый вопрос — проверяем, что вся функция работает от начала до конца
ask_question("Who created the monster in this story?")

ОТВЕТ:
Информация недоступна.

ИСТОЧНИКИ (найденные фрагменты):

[1] *** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Lette...

[2]  being; chord after chord was
sounded, and soon my mind was filled with one thought, one conception,
one purpose. So much has been done, exclaimed the...

[3] same fashion?”

“Yes.”

“Then I fancy we have seen him, for the day before we picked you up we
saw some dogs drawing a sledge, with a man in it, acros...

[4] he science of anatomy, but this was not sufficient; I
must also observe the natural decay and corruption of the human body.
In my education my father ...


In [8]:
# Проверяем вопрос про самое начало книги (письма Уолтона) —
# эта часть точно есть в первых 200 проиндексированных фрагментах
ask_question("Who is Robert Walton writing his letters to?")

ОТВЕТ:
Robert Walton is writing his letters to his sister, Mrs. Saville, in England.

ИСТОЧНИКИ (найденные фрагменты):

[1]  for the present to write to
me by every opportunity: I may receive your letters on some occasions when
I need them most to support my spirits. I love...

[2] t I may again and again testify my gratitude for all your
love and kindness.

Your affectionate brother,

R. Walton




Letter 2

_To Mrs. Saville, En...

[3]  Wherefore not? Thus far I
have gone, tracing a secure way over the pathless seas, the very stars
themselves being witnesses and testimonies of my tri...

[4] 19
 Chapter 20
 Chapter 21
 Chapter 22
 Chapter 23
 Chapter 24




Letter 1

_To Mrs. Saville, England._


St. Petersburgh, Dec. 11th, 17—.


You will...


In [9]:
# Ещё один вопрос про ранние главы — проверяем поиск на знакомой части текста
ask_question("What books did young Victor Frankenstein enjoy reading as a child?")

ОТВЕТ:
Young Victor Frankenstein enjoyed reading the works of poets, as well as the writings of Paracelsus and Albertus Magnus.

ИСТОЧНИКИ (найденные фрагменты):

[1] *** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Lette...

[2] he voyages made for purposes of discovery composed the
whole of our good Uncle Thomas’ library. My education was neglected,
yet I was passionately fon...

[3] ossible that the train of my ideas would never
have received the fatal impulse that led to my ruin. But the cursory glance
my father had taken of my v...

[4] del of nature, and rashly and
ignorantly I had repined.

But here were books, and here were men who had penetrated deeper and knew
more. I took their ...


In [10]:
# Вопрос про персонажа, который тоже упоминается в начале книги
ask_question("Who is Elizabeth Lavenza?")

ОТВЕТ:
Elizabeth Lavenza is the beautiful and adored companion of the narrator's occupations and pleasures, who became the inmate of the narrator's parents' house. She is the daughter of a Milanese nobleman and was raised by rustic guardians after her mother died giving birth to her.

ИСТОЧНИКИ (найденные фрагменты):

[1]  Lavenza
became the inmate of my parents’ house—my more than
sister—the beautiful and adored companion of all my occupations and
my pleasures.

Everyo...

[2] ls. The apparition was soon explained. With his
permission my mother prevailed on her rustic guardians to yield their
charge to her. They were fond of...

[3] n all her features.

The peasant woman, perceiving that my mother fixed eyes of wonder and
admiration on this lovely girl, eagerly communicated her hi...

[4]  symptoms, and the
looks of her medical attendants prognosticated the worst event. On her
deathbed the fortitude and benignity of this best of women d...


In [11]:
import chromadb

# создаём НОВУЮ базу с нуля, чтобы не было конфликтов со старыми 200 фрагментами
chroma_client = chromadb.Client()

# удаляем старую коллекцию, если она уже существует
try:
    chroma_client.delete_collection(name="frankenstein")
except:
    pass

collection = chroma_client.create_collection(name="frankenstein")

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# теперь берём ВСЕ фрагменты, а не первые 200 — чтобы система "видела" всю книгу
all_chunks = chunks

for i, chunk in enumerate(all_chunks):
    embedding = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"source": filename, "chunk_index": i}]
    )
    if (i + 1) % 100 == 0:
        print(f"Обработано {i + 1} из {len(all_chunks)} фрагментов")

print("Готово! Вся книга добавлена в векторную базу.")

Обработано 100 из 1049 фрагментов
Обработано 200 из 1049 фрагментов
Обработано 300 из 1049 фрагментов
Обработано 400 из 1049 фрагментов
Обработано 500 из 1049 фрагментов
Обработано 600 из 1049 фрагментов
Обработано 700 из 1049 фрагментов
Обработано 800 из 1049 фрагментов
Обработано 900 из 1049 фрагментов
Обработано 1000 из 1049 фрагментов
Готово! Вся книга добавлена в векторную базу.


In [12]:
# Повторяем тот же вопрос, что и раньше — теперь, когда вся книга проиндексирована,
# ответ должен найтись (глава про создание монстра теперь тоже в базе)
ask_question("Who created the monster in this story?")

ОТВЕТ:
Монстра в этой истории создал Франкенштейн.

ИСТОЧНИКИ (найденные фрагменты):

[1] owever earnest and connected. Such a monster has, then, really existence!
I cannot doubt it, yet I am lost in surprise and admiration. Sometimes I
end...

[2] o wretched that the light of
day will be hateful to you. You are my creator, but I am your master;
obey!”

“The hour of my irresolution is past, and t...

[3] shore. I inquired of the inhabitants concerning the
fiend and gained accurate information. A gigantic monster, they said,
had arrived the night before...

[4] ed to change, and I thought that I
held the corpse of my dead mother in my arms; a shroud enveloped her
form, and I saw the grave-worms crawling in th...


In [13]:
# Критерий приёмки №1 и №2 из ТЗ: проверка корректности поиска и достоверности ответа
ask_question("What was the name of Victor Frankenstein's father?")

ОТВЕТ:
Информация недоступна.

ИСТОЧНИКИ (найденные фрагменты):

[1] d, but with kindness and affection for those who love you, and not
with hatred for your enemies.

“Your affectionate and afflicted father,

“Alphonse ...

[2] . This also was my doing! And my
father’s woe, and the desolation of that late so smiling home all was
the work of my thrice-accursed hands! Ye weep, ...

[3] l my papa.’

“‘Boy, you will never see your father again; you must come with me.’

“‘Hideous monster! Let me go. My papa is a syndic—he is M.
Frankens...

[4] ow you left
my father, brothers, and Elizabeth.”

“Very well, and very happy, only a little uneasy that they hear from
you so seldom. By the by, I mea...


In [14]:
# Критерий приёмки №3: вопрос, ответа на который в книге точно нет —
# система должна честно сказать "информация недоступна", а не выдумывать
ask_question("What is the capital of France?")

ОТВЕТ:
Информация недоступна.

ИСТОЧНИКИ (найденные фрагменты):

[1]  the
beauty of the scene, sometimes on one side of the lake, where we saw
Mont Salêve, the pleasant banks of Montalègre, and at a distance,
surmountin...

[2] ge at the distance of half a league from the city. The sky
was serene; and, as I was unable to rest, I resolved to visit the spot
where my poor Willia...

[3] of piny mountains, the
impetuous Arve, and cottages every here and there peeping forth from
among the trees formed a scene of singular beauty. But it ...

[4] gly
sister, Manon, married M. Duvillard, the rich banker, last autumn. Your
favourite schoolfellow, Louis Manoir, has suffered several misfortunes
sin...


In [15]:
# Критерий приёмки №5 (воспроизводимость), запуск №1 —
# сравниваем этот ответ со следующим запуском того же вопроса
ask_question("Who created the monster in this story?")

ОТВЕТ:
Создателем монстра в этой истории является Франкенштейн.

ИСТОЧНИКИ (найденные фрагменты):

[1] owever earnest and connected. Such a monster has, then, really existence!
I cannot doubt it, yet I am lost in surprise and admiration. Sometimes I
end...

[2] o wretched that the light of
day will be hateful to you. You are my creator, but I am your master;
obey!”

“The hour of my irresolution is past, and t...

[3] shore. I inquired of the inhabitants concerning the
fiend and gained accurate information. A gigantic monster, they said,
had arrived the night before...

[4] ed to change, and I thought that I
held the corpse of my dead mother in my arms; a shroud enveloped her
form, and I saw the grave-worms crawling in th...


In [16]:
# Критерий приёмки №5 (воспроизводимость), запуск №2 —
# при temperature=0 ответ должен совпадать с предыдущим запуском
ask_question("Who created the monster in this story?")

ОТВЕТ:
Создателем монстра в этой истории является Франкенштейн.

ИСТОЧНИКИ (найденные фрагменты):

[1] owever earnest and connected. Such a monster has, then, really existence!
I cannot doubt it, yet I am lost in surprise and admiration. Sometimes I
end...

[2] o wretched that the light of
day will be hateful to you. You are my creator, but I am your master;
obey!”

“The hour of my irresolution is past, and t...

[3] shore. I inquired of the inhabitants concerning the
fiend and gained accurate information. A gigantic monster, they said,
had arrived the night before...

[4] ed to change, and I thought that I
held the corpse of my dead mother in my arms; a shroud enveloped her
form, and I saw the grave-worms crawling in th...
